In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import celltypist
from celltypist import models
from celltypist.models import Model

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/GSE202379_Raw.h5ad")
adata

In [ ]:
adata.raw = adata
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
# Scale data
sc.pp.scale(adata, max_value=10)
# Run PCA
sc.tl.pca(adata, svd_solver='arpack')
print(adata.obsm['X_pca'][:5, :5])
sc.pl.pca(adata, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['Disease_status'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata.obsm['X_pca'],adata.obs, 'sample')

# Harmony 결과를 저장
adata.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata)
sc.pl.umap(adata, color=['Disease_status'])

In [ ]:
sc.tl.leiden(adata, resolution=0.27, key_added="cellannot")
sc.pl.umap(adata, color=['cellannot'])

In [ ]:
sc.pl.umap(
    adata,
    color='cellannot',
    frameon=False,
    title='cellannot',
    size=10
)

In [ ]:
# Hepatocyte annot
sc.pl.dotplot(adata,['ASGR1','TFR2','APOC1','ADH4','HPX','C3','FGB','PLG','FYB2','APOB','GHR','TENM2','CYP3A4','CPS1','ASS1','GLUL'], groupby="cellannot", standard_scale='var')

In [ ]:
adata.raw

In [ ]:
rank_df = sc.get.rank_genes_groups_df(adata, group=None)
rank_df

In [ ]:
adata.obs['cellannot'] = adata.obs['cellannot'].replace({
    '0': 'Hepatocyte',
    '1': 'Hepatocyte',
    '2': 'Hepatocyte',
    '3': 'Endothelial',
    '4': 'Cholangiocyte',
    '5': 'Hepatocyte',
    '6': 'Lymphocyte',
    '7': 'Stellete',
    '8': 'Macrophage',
    '9': 'Hepatocyte',
    '10': 'B cells'

})
sc.pl.umap(
    adata,
    color='cellannot',
    frameon=False,
    title='cellannot',
    size=10
)

In [ ]:
sc.tl.dendrogram(adata,groupby='cellannot')
sc.tl.rank_genes_groups(adata, groupby='cellannot', method='wilcoxon',use_raw=True)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10)

In [ ]:
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata,
    var_names=top_genes,
    groupby="cellannot",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
adata.write("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/GSE202379_cellannot.h5ad")

In [ ]:
# 원하는 순서 정의
desired_order = ["Healthy control", "MASLD", "MASH w/o cirrhosis", "MASH with cirrhosis", "end stage" ]
adata.obs['Disease_status'] = pd.Categorical(adata.obs['Disease_status'], categories=desired_order, ordered=True)

cluster_labels = adata.obs['cellannot']
batch_labels = adata.obs['Disease_status']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('MASLD stage cell lineages proportion')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='Disease state cellannot', bbox_to_anchor=(1.05, 1), loc='upper left')